# ECA-LDNet reviewer evaluation

This notebook evaluates the 3 full-model seeds and the 12 available Stage 4 ablation checkpoints. Attach the package ZIP and evaluation datasets in Kaggle. Enable a GPU and Internet in notebook settings; Internet is needed to install optional metrics and fetch their pretrained weights. The training datasets are not needed for the final inference evaluation.

Paired datasets are scored at 256×256 for synthetic tests and native resolution for real paired tests. The evaluator saves all successfully matched and scored predictions under `tested_sample_<dataset>` and writes per-image CSV, summary CSV, and provenance JSON. It does not cherry-pick images based on their scores.

In [ ]:
%pip -q install 'keras>=3.9,<4' pyiqa scikit-image fade-python
print('Optional paired perceptual metrics (LPIPS/DISTS), CIEDE2000, and FADE dependencies installed.')

In [ ]:
from pathlib import Path
import json, os, shutil, sys, zipfile

INPUTS_ROOT = Path('/kaggle/input')
ZIP_HITS = list(INPUTS_ROOT.rglob('eca_ldnet_reviewer_test_package.zip'))
MANIFEST_HITS = list(INPUTS_ROOT.rglob('package_manifest.json'))
if len(ZIP_HITS) == 1:
    PACKAGE_ARCHIVE = ZIP_HITS[0]
    EXTRACT_ROOT = Path('/kaggle/working/eca_ldnet_reviewer_package')
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PACKAGE_ARCHIVE) as zf:
        bad = zf.testzip()
        assert bad is None, f'Corrupt package entry: {bad}'
        root = EXTRACT_ROOT.resolve()
        for member in zf.infolist():
            target = (EXTRACT_ROOT / member.filename).resolve()
            assert target == root or root in target.parents, f'Unsafe ZIP path: {member.filename}'
        zf.extractall(EXTRACT_ROOT)
    PACKAGE_ROOT = EXTRACT_ROOT / 'reviewer_test_package'
elif len(MANIFEST_HITS) == 1:
    PACKAGE_ROOT = MANIFEST_HITS[0].parent
else:
    raise RuntimeError(f'Expected one uploaded package ZIP or extracted package manifest; ZIPs={ZIP_HITS}, manifests={MANIFEST_HITS}')

CODE_ROOT = PACKAGE_ROOT / 'code'
MODEL_MANIFEST = json.loads((PACKAGE_ROOT / 'package_manifest.json').read_text())
assert len(MODEL_MANIFEST['models']) == 15, MODEL_MANIFEST['counts']
sys.path.insert(0, str(CODE_ROOT))
os.environ['PYTHONPATH'] = str(CODE_ROOT)
print('Package:', PACKAGE_ROOT)
print('Models:', MODEL_MANIFEST['counts'])
for model in MODEL_MANIFEST['models']:
    checkpoint = PACKAGE_ROOT / model['checkpoint']
    assert checkpoint.is_file(), checkpoint
    print(model['model_id'], checkpoint.stat().st_size / 1e6, 'MB')

import tensorflow as tf
import keras
from packaging.version import Version
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
assert Version(keras.__version__) >= Version('3.9'), 'Install Keras >=3.9 before loading these checkpoints.'
print('GPU:', tf.config.list_physical_devices('GPU'))
GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Enable a Kaggle GPU before evaluation.'
for gpu in GPUS:
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

# Smoke-test every saved model and its dynamic-resolution rebuild before long evaluation.
import gc, numpy as np
from eca_ldnet.model import load_pretrained, build_eca_ldnet
VARIANT_FLAGS = {'full': {}, 'no_eca': {'use_eca': False},
                 'eca_only': {'use_eca': True, 'use_pa': False},
                 'no_attention': {'use_eca': False, 'use_pa': False},
                 'no_physics': {'use_physics': False}}
for spec in MODEL_MANIFEST['models']:
    fixed = load_pretrained(str(PACKAGE_ROOT / spec['checkpoint']), compile=False)
    has_physics = 'physics_output' in {layer.name for layer in fixed.layers}
    observed = len(fixed.get_layer('physics_output').input) == 4 if has_physics else True
    dynamic = build_eca_ldnet(img_size=None, physics_uses_observed_input=observed,
                              **VARIANT_FLAGS[spec['variant']])
    dynamic.set_weights(fixed.get_weights())
    for height, width in [(256, 256), (272, 304)]:
        output = dynamic(np.zeros((1, height, width, 3), np.float32), training=False)
        assert tuple(output.shape) == (1, height, width, 3), (spec['model_id'], output.shape)
    print('Reload + dynamic inference OK:', spec['model_id'], 'parameters:', dynamic.count_params())
    del fixed, dynamic, output
    tf.keras.backend.clear_session(); gc.collect()
print('All 15 checkpoints passed the pre-evaluation load/shape smoke test.')

## Attach and map datasets

The two paths below are the Kaggle mounts already used in the training/evaluation runbooks. RESIDE-6K is added only when a separate hazy/GT test split exists; training folders are never used as test data. Edit `REAL_PAIRED_PATHS` and `RTTS_HAZY` to match the datasets you attach. The official source links and dataset names are in `REVIEWER_TEST_PACKAGE_README.md`.

In [ ]:
from pathlib import Path

RESIDE_TEST_ROOT = Path('/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/test')
SOTS_ROOT = Path('/kaggle/input/datasets/balraj98/synthetic-objective-testing-set-sots-reside')

def image_dir_pair(hazy, gt):
    return Path(hazy).is_dir() and Path(gt).is_dir()

PAIRED_DATASETS = []
# Include only a genuinely separate held-out test split.
reside_hazy, reside_gt = RESIDE_TEST_ROOT / 'hazy', RESIDE_TEST_ROOT / 'GT'
if image_dir_pair(reside_hazy, reside_gt):
    PAIRED_DATASETS.append({'name': 'RESIDE-6K-Test', 'hazy_dir': str(reside_hazy), 'gt_dir': str(reside_gt), 'size': 256})
else:
    print('RESIDE-6K test pair folders absent; skipping (do not substitute training images).')

for label, relative in [('SOTS-Indoor', 'indoor'), ('SOTS-Outdoor', 'outdoor')]:
    hazy, gt = SOTS_ROOT / relative / 'hazy', SOTS_ROOT / relative / 'clear'
    if image_dir_pair(hazy, gt):
        PAIRED_DATASETS.append({'name': label, 'hazy_dir': str(hazy), 'gt_dir': str(gt), 'size': 256})
    else:
        print('SOTS folders absent; fix SOTS_ROOT/layout or skip:', hazy, gt)

# Fill each pair below after attaching that public dataset. Leave None to skip.
REAL_PAIRED_PATHS = {
    'O-HAZE':   (None, None),
    'I-HAZE':   (None, None),
    'NH-HAZE':  (None, None),
    'DNH-HAZE': (None, None),
}
for name, paths in REAL_PAIRED_PATHS.items():
    if paths[0] is None or paths[1] is None:
        print('Not attached/configured; skip:', name)
        continue
    hazy, gt = map(Path, paths)
    if image_dir_pair(hazy, gt):
        PAIRED_DATASETS.append({'name': name, 'hazy_dir': str(hazy), 'gt_dir': str(gt), 'native_resolution': True})
    else:
        print('Invalid real paired paths; correct these before running:', name, hazy, gt)

# RTTS has no clean reference; attach RESIDE-β and point to its JPEGImages folder.
RTTS_HAZY = None  # Example: Path('/kaggle/input/.../RTTS/JPEGImages')
UNPAIRED_DATASETS = []
if RTTS_HAZY is not None and Path(RTTS_HAZY).is_dir():
    UNPAIRED_DATASETS.append({'name': 'RTTS', 'hazy_dir': str(Path(RTTS_HAZY))})

assert PAIRED_DATASETS or UNPAIRED_DATASETS, 'Attach at least SOTS or another evaluation dataset and update its path.'
DATASET_CONFIG = Path('/kaggle/working/reviewer_dataset_config.json')
DATASET_CONFIG.write_text(json.dumps({'paired': PAIRED_DATASETS, 'unpaired': UNPAIRED_DATASETS}, indent=2))
print('Paired datasets:', json.dumps(PAIRED_DATASETS, indent=2))
print('Unpaired datasets:', json.dumps(UNPAIRED_DATASETS, indent=2))

## Optional validation-only blend and epsilon sweep

This uses the packaged seed 17 validation split and the six blend / three epsilon values requested by the reviewers. It runs only when the original ITS and RESIDE training inputs are attached at the paths recorded in the split manifest. It never reads SOTS or the real test sets for parameter selection. The sweep reports PSNR/SSIM and bootstrap intervals; it is an inference screening study, not a retrained learnable-blend comparison.

In [ ]:
import csv, subprocess, sys

full17 = next(m for m in MODEL_MANIFEST['models'] if m['model_id'] == 'full_seed17')
SPLIT_MANIFEST = PACKAGE_ROOT / full17['split_manifest']
with SPLIT_MANIFEST.open(newline='', encoding='utf-8-sig') as f:
    validation_rows = [r for r in csv.DictReader(f) if r.get('split') == 'validation']
from eca_ldnet.data import clean_scene_id, get_gt_path
TRAIN_DIR_PAIRS = [(Path('/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/train/hazy'),
                    Path('/kaggle/input/datasets/kmljts/reside-6k/RESIDE-6K/train/GT')),
                   (Path('/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard/hazy'),
                    Path('/kaggle/input/datasets/balraj98/indoor-training-set-its-residestandard/clear'))]
sweep_pairs, missing_validation_paths = [], []
for row in validation_rows:
    found = None
    for hazy_dir, gt_dir in TRAIN_DIR_PAIRS:
        candidate = hazy_dir / Path(row['hazy_path']).name
        target = get_gt_path(candidate.name, str(gt_dir)) if candidate.is_file() and gt_dir.is_dir() else None
        if target and clean_scene_id(target) == row['clean_sha256']:
            found = (str(candidate), target); break
    if found: sweep_pairs.append({'hazy_path': found[0], 'clean_path': found[1], 'clean_sha256': row['clean_sha256'], 'split': 'validation'})
    else: missing_validation_paths.append(row['hazy_path'])
RELOCATED_VALIDATION = Path('/kaggle/working/reviewer_validation_seed17.csv')
if sweep_pairs:
    with RELOCATED_VALIDATION.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['hazy_path','clean_path','clean_sha256','split']); writer.writeheader(); writer.writerows(sweep_pairs)
RUN_VALIDATION_BLEND_SWEEP = bool(validation_rows) and not missing_validation_paths
if RUN_VALIDATION_BLEND_SWEEP:
    sweep_out = Path('/kaggle/working/reviewer_evaluation/validation_sweeps/seed17_blend_epsilon')
    sweep_cmd = [sys.executable, str(CODE_ROOT / 'scripts/sweep_asm_blend.py'),
                 '--model', str(PACKAGE_ROOT / full17['checkpoint']),
                 '--manifest', str(RELOCATED_VALIDATION), '--split', 'validation',
                 '--preprocessing', 'radiometric', '--output', str(sweep_out),
                 '--weights', '0', '0.02', '0.05', '0.08', '0.10', '0.20',
                 '--epsilons', '0.05', '0.10', '0.20', '--bootstrap-samples', '5000', '--seed', '42']
    subprocess.run(sweep_cmd, check=True)
else:
    print('Skipped validation sweep: attach ITS and RESIDE-6K training inputs or correct TRAIN_DIR_PAIRS. Matched validation pairs:', len(sweep_pairs), '/', len(validation_rows), 'missing:', len(missing_validation_paths))

## Run all models and save evidence

By default, this runs every packaged checkpoint on every enabled dataset. Each finished model/dataset task is checkpointed in `progress.json`; rerunning the cell with `--resume` skips tasks that already have complete run manifests. The outputs ZIP contains paired/unpaired summaries, per-image CSVs, hashes/configuration JSON, and all `tested_sample_<dataset>` predictions.

In [ ]:
import subprocess, sys

RESULTS_ROOT = Path('/kaggle/working/reviewer_evaluation')
cmd = [sys.executable, str(CODE_ROOT / 'scripts/run_reviewer_evaluation.py'),
       '--package-root', str(PACKAGE_ROOT),
       '--dataset-config', str(DATASET_CONFIG),
       '--output', str(RESULTS_ROOT),
       '--perceptual', 'lpips', 'dists',
       '--nr-metrics', 'niqe', 'brisque', 'musiq', 'maniqa', 'clipiqa',
       '--bootstrap-samples', '5000', '--seed', '42', '--resume']
if UNPAIRED_DATASETS:
    cmd.append('--fade')
print('Running', len(MODEL_MANIFEST['models']), 'models on', len(PAIRED_DATASETS), 'paired and', len(UNPAIRED_DATASETS), 'unpaired datasets')
subprocess.run(cmd, check=True)
RESULTS_ZIP = Path('/kaggle/working/reviewer_evaluation_results.zip')
assert RESULTS_ZIP.is_file(), RESULTS_ZIP
print('Downloadable results:', RESULTS_ZIP, f'{RESULTS_ZIP.stat().st_size / 1e9:.2f} GB')

In [ ]:
# Reviewer deployment benchmark: batch 1, same Kaggle GPU/software, four target sizes.
RUN_RESOLUTION_BENCHMARK = True
if RUN_RESOLUTION_BENCHMARK:
    benchmark_cmd = [sys.executable, str(CODE_ROOT / 'scripts/benchmark_reviewer_models.py'),
                     '--package-root', str(PACKAGE_ROOT),
                     '--output', str(RESULTS_ROOT / 'resolution_latency_memory.csv'),
                     '--resolutions', '256x256', '512x512', '1280x720', '1920x1080',
                     '--warmup', '10', '--runs', '50']
    subprocess.run(benchmark_cmd, check=True)
    # Refresh the downloadable archive to include the benchmark CSV/JSON.
    shutil.make_archive('/kaggle/working/reviewer_evaluation_results', 'zip', root_dir=RESULTS_ROOT)
    print('Updated results ZIP with deployment benchmark.')
else:
    print('Skipped resolution benchmark; no latency/memory evidence was added.')

In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(RESULTS_ZIP)))
print('Top-level summary files:')
for path in sorted(RESULTS_ROOT.glob('*.csv')) + sorted(RESULTS_ROOT.glob('*.json')):
    print(path.name, path.stat().st_size)